In [1]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import functions
from FNO import *
import pybamm
import flax

In [2]:
anode_file = "trained_models/diff_D/anode_GRF__2025-03-27_12-15-16.msgpack"
cathode_file = "trained_models/diff_D/cathode_GRF__2025-03-27_13-13-25.msgpack"

train_data = np.load("data/spm/for_soc/test/GRF_2200.npz")
test_data = np.load("data/spm/for_soc/test/GRF_2200.npz")

In [3]:
train_I = jnp.array(train_data["current"])
test_I = jnp.array(test_data["current"])

train_cn_anode = jnp.array(train_data["target_concentration_anode"])
test_cn_anode = jnp.array(test_data["target_concentration_anode"])

train_c0_anode = jnp.array(train_data["initial_concentration_anode"])
test_c0_anode = jnp.array(test_data["initial_concentration_anode"])

train_cn_cathode = jnp.array(train_data["target_concentration_cathode"])
test_cn_cathode = jnp.array(test_data["target_concentration_cathode"])

train_c0_cathode = jnp.array(train_data["initial_concentration_cathode"])
test_c0_cathode = jnp.array(test_data["initial_concentration_cathode"])

cpu = jax.devices("cpu")[0]
gpu = jax.devices("gpu")[0]  # Should be the GPU

In [4]:
params_anode = functions.load_model_params(anode_file)
params_cathode = functions.load_model_params(cathode_file)

In [5]:
# Assume these hyperparameters
k_modes = 10
fno_depth = 6
hidden_channels = 64
output_channels = 1

# Padding amounts
padding_t = 5  # along t-axis
padding_r = 2  # along r-axis

# Original sample counts
num_samples_I = 75
num_samples_c0 = 20

# Create dummy inputs
key_main = jax.random.PRNGKey(0)
dummy_I = jax.random.normal(key_main, (num_samples_I,))  # shape (75,)
dummy_c0 = jax.random.normal(key_main, (num_samples_c0,)) # shape (20,)

# Choose one sample for I and c0
I_single = dummy_I[0]   # scalar
c0_single = dummy_c0[0] # scalar

# Original resolutions without padding
H_orig = 20
W_orig = 75

H = H_orig + 2 * padding_r  # 20 + 2*2 = 24
W = W_orig + 2 * padding_t  # 75 + 2*5 = 85

# Pad I along t-axis (turning a scalar into a vector of length W)
# Here I_single is a scalar, we need a (W_orig,) vector. Let's assume we want a constant I for all t.
# For a single dummy run, let's just create a constant vector from I_single:
I_vector = jnp.full((W_orig,), I_single)
I_padded = jnp.pad(I_vector, (padding_t, padding_t))  # shape (85,)

# Pad c0 along r-axis (similarly create a vector from c0_single)
c0_vector = jnp.full((H_orig,), c0_single)
c0_padded = jnp.pad(c0_vector, (padding_r, padding_r))  # shape (24,)

# Create coordinate grids
r_lin = jnp.linspace(0, 1, H)  # shape (24,)
t_lin = jnp.linspace(0, 1, W)  # shape (85,)
R, T = jnp.meshgrid(r_lin, t_lin, indexing='ij')  # R,T both (24,85)

# Broadcast I_padded (85,) to (24,85)
I_2D = jnp.tile(I_padded[None, :], (H, 1))   # (24,85)
# Broadcast c0_padded (24,) to (24,85)
c0_2D = jnp.tile(c0_padded[:, None], (1, W)) # (24,85)

# Stack all channels: (B,H,W,C) with B=1
input_tensor = jnp.stack([I_2D, c0_2D, R, T], axis=-1)   # (24,85,4)
input_tensor = input_tensor[None, ...]  # Add batch dimension: (1,24,85,4)

# Instantiate model
model = FNO(k_modes=k_modes, fno_depth=fno_depth, hidden_channels=hidden_channels, output_channels=output_channels)

# Initialize parameters
init_key = jax.random.PRNGKey(42)
params = model.init(init_key, input_tensor)

# Forward pass
anode = model.apply(params, input_tensor)
cathode = model.apply(params, input_tensor)
print("Output shape:", anode.shape)  # should be (1,24,85,1)
print("Output shape:", cathode.shape)

Output shape: (1, 24, 85, 1)
Output shape: (1, 24, 85, 1)


In [6]:
def preprocess_data(train_I, train_c0, train_cn):
    """
    Pre-process the dataset into channels and padded shapes suitable for the FNO.
    
    Parameters
    ----------
    train_I : np.ndarray
        Current samples of shape (num_samples, 75).
    train_c0 : np.ndarray
        Initial concentration samples of shape (num_samples, 20).
    train_cn : np.ndarray
        Target concentrations of shape (num_samples, 20, 75).

    Returns
    -------
    X : jnp.ndarray
        Preprocessed inputs of shape (num_samples, 24, 85, 4).
    Y : jnp.ndarray
        Preprocessed targets of shape (num_samples, 24, 85, 1).
    """

    num_samples = train_I.shape[0]
    t_lin = t/t_max
    R_orig, T_orig = jnp.meshgrid(r, t_lin, indexing='ij')  # (20,75) each

    R =jnp.pad(R_orig, ((padding_r, padding_r),(padding_t, padding_t)))
    T = jnp.pad(T_orig, ((padding_r, padding_r),(padding_t, padding_t)))
    # R, T now have shape (24,85)


    # Prepare arrays for output
    X = jnp.zeros((num_samples, H, W, 4))
    Y = jnp.zeros((num_samples, H, W, 1))

    # Pad function
    def pad_along_time(arr, padding):
        # arr shape is (..., 75)
        return jnp.pad(arr, ((0,0), (padding, padding)))

    def pad_along_r(arr, padding):
        # arr shape is (..., 20)
        return jnp.pad(arr, ((0,0), (padding, padding)))

    # Pad I and c0 along appropriate axes
    # For each sample:
    # train_I[n]: shape (75,) -> pad to (85,)
    # train_c0[n]: shape (20,) -> pad to (24,)

    I_padded = pad_along_time(train_I, padding_t)    # (num_samples, 85)
    c0_padded = pad_along_r(train_c0, padding_r)      # (num_samples, 24)

    # Broadcast to (24,85) for each sample
    # I_padded: (num_samples, 85) -> (num_samples,24,85)
    # replicate along r-dim
    I_2D = jnp.tile(I_padded[:, None, :], (1, H, 1))

    # c0_padded: (num_samples,24) -> (num_samples,24,85)
    # replicate along t-dim
    c0_2D = jnp.tile(c0_padded[:, :, None], (1, 1, W))

    # R,T are the same for all samples, just expand
    R_3D = jnp.tile(R[None, ...], (num_samples, 1, 1)) # (num_samples,24,85)
    T_3D = jnp.tile(T[None, ...], (num_samples, 1, 1)) # (num_samples,24,85)

    # Stack channels: (num_samples,24,85,4)
    X = jnp.stack([I_2D, c0_2D, R_3D, T_3D], axis=-1)

    # Process targets (train_cn): (num_samples,20,75)
    # pad targets along r and t to (24,85)
    # Along r: pad (20,) to (24,)
    # Along t: pad (75,) to (85,)

    # Pad cn along both dimensions:
    # We'll do this per-sample:
    cn_padded = jnp.pad(train_cn, ((0,0),(padding_r,padding_r),(padding_t,padding_t))) # (num_samples,24,85)

    # Add channel dimension for targets: (num_samples,24,85,1)
    Y = cn_padded[..., None]

    return X,Y#jax.device_put(X, cpu), jax.device_put(Y, cpu)

In [7]:
# Set random keys
main_key = jax.random.PRNGKey(0)
key_train, key_test = jax.random.split(main_key)

params_bat = pybamm.ParameterValues("Chen2020")

C = params_bat["Nominal cell capacity [A.h]"]
Dan = params_bat["Negative particle diffusivity [m2.s-1]"]
Dca = params_bat["Positive particle diffusivity [m2.s-1]"]
Ran = params_bat["Negative particle radius [m]"]
Rca = params_bat["Positive particle radius [m]"]
epsan = params_bat["Negative electrode active material volume fraction"]
epsca = params_bat["Positive electrode active material volume fraction"]
cs_max_a = params_bat["Maximum concentration in negative electrode [mol.m-3]"]
cs_max_c = params_bat["Maximum concentration in positive electrode [mol.m-3]"]
Lan = params_bat["Negative electrode thickness [m]"]
Lca = params_bat["Positive electrode thickness [m]"]
A = params_bat["Electrode height [m]"] * params_bat["Electrode width [m]"]
t_max = 900
num_samples_I = 75
num_samples_c0 = 20

t = np.linspace(0, t_max, num_samples_I)
r = np.linspace(0, 1, num_samples_c0)

In [8]:
params_anode = flax.serialization.from_bytes(params, params_anode)
params_cathode = flax.serialization.from_bytes(params, params_cathode)

In [9]:
train_idx = np.random.randint(2, 88000)
test_idx = np.random.randint(2, 8800)

In [ ]:
def get_dummy_measurement(I, cn_anode, cn_cathode, noise = 0, max_retries = 1000):

    for _ in range(max_retries):

        test_idx = np.random.randint(0, 8800)
        I = I[test_idx]
        cn_anode = cn_anode[test_idx]
        cn_cathode = cn_cathode[test_idx]

        _, V_true = functions.post_proc(params_bat, I, 0, cn_anode[-1,:], 0, cn_cathode[-1,:], Ran, Rca, epsan, epsca, Lan, Lca, A)
        
        if jnp.isnan(V_true).any():
            continue
        else:
            return V_true + noise, test_idx
    
    raise ValueError("Could not produce a NaN-free measurement after {} retries.".format(max_retries))

In [93]:
def U_OCP_an(sto):
    u_eq = (
        1.9793 * jnp.exp(-39.3631 * sto)
        + 0.2482
        - 0.0909 * jnp.tanh(29.8538 * (sto - 0.1234))
        - 0.04478 * jnp.tanh(14.9159 * (sto - 0.2769))
        - 0.0205 * jnp.tanh(30.4444 * (sto - 0.6103))
    )

    return u_eq

def U_OCP_ca(sto):
    u_eq = (
        -0.8090 * sto
        + 4.4875
        - 0.0428 * jnp.tanh(18.5138 * (sto - 0.5542))
        - 17.7326 * jnp.tanh(15.7890 * (sto - 0.3117))
        + 17.5842 * jnp.tanh(15.9308 * (sto - 0.3120))
    )

    return u_eq


    return u_eq


In [94]:
# U_OCP_an = params_bat["Negative electrode OCP [V]"]
# U_OCP_ca = params_bat["Positive electrode OCP [V]"]
R_gas = params_bat['Ideal gas constant [J.K-1.mol-1]']
F = params_bat['Faraday constant [C.mol-1]']
Temp = params_bat["Ambient temperature [K]"]

In [95]:
def in_arcsinh(I, R, epsilon, L, A):

    x = I * R / (3 * epsilon * L * A)

    return x

In [96]:

t_lin = t/t_max
R_orig, T_orig = jnp.meshgrid(r, t_lin, indexing='ij')  # (20,75) each

R =jnp.pad(R_orig, ((padding_r, padding_r),(padding_t, padding_t)))
T = jnp.pad(T_orig, ((padding_r, padding_r),(padding_t, padding_t)))

In [97]:
from scipy.optimize import minimize

In [98]:
def preprocess_candidates(I_array, c0_array):
    """
    Convert a single (or batch of) current and c0 arrays into
    padded 2D channels for the FNO: shape (B, 24, 85, 4).
    
    Parameters
    ----------
    I_array : jnp.ndarray
        If shape is (75,) => a single sample. Or (B,75) => batch.
    c0_array : jnp.ndarray
        If shape is (20,) => a single sample. Or (B,20) => batch.
    
    Returns
    -------
    X : jnp.ndarray
        FNO input, shape (B, 24, 85, 4).
        Where channels are: [I_2D, c0_2D, R_padded, T_padded].
    """
    # 1) Ensure we have a batch dimension. If user passes shape (75,), add batch=1
    if I_array.ndim == 1:
        I_array = I_array[None, :]   # shape (1,75)
    if c0_array.ndim == 1:
        c0_array = c0_array[None, :] # shape (1,20)
    
    B = I_array.shape[0]  # batch size
    
    # 2) Pad along t-axis => shape (B, 75+10) = (B,85)
    I_padded = jnp.pad(I_array, ((0, 0), (padding_t, padding_t)))  # (B,85)
    # 3) Pad along r-axis => shape (B, 20+4) = (B,24)
    c0_padded = jnp.pad(c0_array, ((0, 0), (padding_r, padding_r))) # (B,24)
    
    # 4) Broadcast to (B, 24,85)
    I_2D = jnp.tile(I_padded[:, None, :], (1, H, 1))       # shape (B,24,85)
    c0_2D = jnp.tile(c0_padded[:, :, None], (1, 1, W))     # shape (B,24,85)
    
    # 5) Broadcast R_padded, T_padded => shape (B,24,85)
    #    They are constant, so just replicate along batch dimension
    R_3D = jnp.tile(R[None, ...], (B, 1, 1))  # (B,24,85)
    T_3D = jnp.tile(T[None, ...], (B, 1, 1))  # (B,24,85)
    
    # 6) Stack channels => shape (B, 24,85,4)
    X = jnp.stack([I_2D, c0_2D, R_3D, T_3D], axis=-1)
    
    return X

In [99]:
def postprocess_candidates(c_pred: jnp.ndarray,) -> jnp.ndarray:
    """
    Post-process FNO output for a single electrode (anode or cathode).
    1. Removes radial/time padding (2 in r, 5 in t).
    2. Extracts the last radial index (surface).
    
    Parameters
    ----------
    c_pred : jnp.ndarray
        FNO predictions of shape (B, 24, 85, 1) for a batch size B (or B=1).

    Returns
    -------
    jnp.ndarray
        Surface concentration at the outer radial index, shape (B, W_orig).
    """
    # Unpad the radial and time dimensions: shape (B, 20, 75)
    c_unpadded = c_pred[
        :,
        padding_r : padding_r + H_orig,
        padding_t : padding_t + W_orig,
        0
    ]
    # Take the last radial index => surface concentration, shape (B, 75)
    c_surface = c_unpadded[:, -1, :]

    return c_surface


In [100]:
@jax.jit
def fno_predict_voltage(c0_anode, c0_cathode, current):
# def post_proc(params, I_c, c_pred_an, c_true_an, c_pred_ca, c_true_ca, Ran, Rca, epsan, epsca, Lan, Lca, A):

    X_anode = preprocess_candidates(current,c0_anode)
    X_cathode = preprocess_candidates(current,c0_cathode)

    cn_anode = model.apply(params_anode,X_anode)
    cn_cathode = model.apply(params_cathode,X_cathode)

    cn_anode_surf = postprocess_candidates(cn_anode)
    cn_cathode_surf = postprocess_candidates(cn_cathode)

    j_anode = cn_anode_surf**0.5 * (1-cn_anode_surf)**0.5
    j_cathode = cn_cathode_surf**0.5 * (1-cn_cathode_surf)**0.5

    xan = in_arcsinh(-current, Ran, epsan, Lan, A)
    xca = in_arcsinh(-current, Rca, epsca, Lca, A)

    V_pred = U_OCP_ca(cn_cathode_surf) - U_OCP_an(cn_anode_surf) - 2 * R_gas*Temp/F * jnp.arcsinh(0.5*xan/(j_anode)) - 2 * R_gas*Temp/F * jnp.arcsinh(0.5*xca/(j_cathode))

    return V_pred

In [101]:
V_pred = fno_predict_voltage(test_c0_anode[:3000], test_c0_cathode[:3000], test_I[:3000])

In [102]:
V_pred.shape

(3000, 75)

In [111]:
######Jetzt gehts los######

#train_idx = np.random.randint(2, 88000)
test_idx = np.random.randint(1, 8800)
batch_size = 1000




V_meas, test_idx = get_dummy_measurement(test_I, test_cn_anode, test_cn_cathode, noise = 0)


(20, 75)


In [126]:
@jax.jit
def objective_fn_forward(c0_anode, c0_cathode, current, V_meas):
    # Predict shape (1,75)
    current_batch = jnp.tile(current[None,...], (batch_size, 1))
    V_meas_batch = jnp.tile(V_meas[None,...], (batch_size, 1))

    V_pred = fno_predict_voltage(c0_anode, c0_cathode, current_batch)
    residuals = V_pred - V_meas_batch  # shape (75,)
    return jnp.mean(residuals**2, axis = 1)#V_pred, V_meas_batch #jnp.mean(residuals**2)

In [127]:
residuals_squared = objective_fn_forward(test_c0_anode[:1000], test_c0_cathode[:1000], test_I[0], V_meas)

In [129]:
residuals_squared

Array([0.06785571, 0.12412941, 0.17217408,        nan, 0.06785571,
       0.0328537 , 0.01329556,        nan, 0.06785571, 0.01593555,
       0.04757723, 0.05293006, 0.06785571, 0.0819317 , 0.00978718,
       0.00533676, 0.06785571, 0.06772049, 0.04999737, 0.06669351,
       0.06785571, 0.01079467, 0.09649459, 0.17215966, 0.06785571,
              nan,        nan,        nan, 0.06785571, 0.0125403 ,
       0.00684996, 0.03874332, 0.06785571, 0.18459256, 0.81043285,
              nan, 0.06785571, 0.04406217, 0.08420634, 0.01773014,
       0.06785571, 0.38395378,        nan,        nan, 0.06785571,
       0.00956039, 0.01525607, 0.07197922, 0.06785571, 0.05832626,
       0.00823214, 0.06734142, 0.06785571, 0.2803601 ,        nan,
              nan, 0.06785571, 0.03763623, 0.00577162, 0.04293572,
       0.06785571, 0.04083897, 0.03195703, 0.04685249, 0.06785571,
       0.05920015, 0.12796558, 0.00791345, 0.06785571, 0.20576642,
              nan,        nan, 0.06785571, 0.00810669, 0.04720